## Two Layer DNN Models

In [1]:
import os
import time
import warnings
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn as sk
import tensorflow as tf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# set current data path
data_path = os.getcwd()
output_dir = Path(data_path) / "multi_layer_results"
output_dir.mkdir(exist_ok=True)

# import kaggle token
token = Path("kaggletoken.txt").read_text().strip()
os.environ["KAGGLE_API_TOKEN"] = token

# define kaggle download
download_path = kagglehub.dataset_download(
    "vincentvaseghi/us-cities-housing-market-data",
    path="city_market_tracker.tsv000",
    output_dir=data_path + "/data",
)

# check rocm devices
devices = tf.config.list_physical_devices("GPU")
print("GPUs found:", devices)
print("Saving outputs to:", output_dir)


/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0000 00:00:1774067945.377157  577429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774067945.377225  577429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774067945.377227  577429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774067945.377228  577429 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-03-21 00:39:05.386455: I tensorflow/core/platform/cpu_f

GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Saving outputs to: /workspace/Math335_Project1/multi_layer_results


## Import into pandas

In [2]:
import pandas as pd

# load the tsv as a csv but with tab seperators
df = pd.read_csv(data_path + "/data/city_market_tracker.tsv000", sep="\t")

# print head of dataset
print(df.head())
df.columns.tolist()


  PERIOD_BEGIN  PERIOD_END  PERIOD_DURATION REGION_TYPE  REGION_TYPE_ID  \
0   2021-08-01  2021-08-31               30       place               6   
1   2022-03-01  2022-03-31               30       place               6   
2   2016-09-01  2016-09-30               30       place               6   
3   2025-12-01  2025-12-31               30       place               6   
4   2018-03-01  2018-03-31               30       place               6   

   TABLE_ID  IS_SEASONALLY_ADJUSTED            REGION          CITY  \
0     14689                   False   St. Augusta, MN   St. Augusta   
1     30390                   False       Newfane, NY       Newfane   
2     18083                   False  Simpsonville, KY  Simpsonville   
3     11261                   False         Maize, KS         Maize   
4     30755                   False    Loganville, GA    Loganville   

       STATE  ... SOLD_ABOVE_LIST_YOY PRICE_DROPS  PRICE_DROPS_MOM  \
0  Minnesota  ...            0.354167    0.272727   

['PERIOD_BEGIN',
 'PERIOD_END',
 'PERIOD_DURATION',
 'REGION_TYPE',
 'REGION_TYPE_ID',
 'TABLE_ID',
 'IS_SEASONALLY_ADJUSTED',
 'REGION',
 'CITY',
 'STATE',
 'STATE_CODE',
 'PROPERTY_TYPE',
 'PROPERTY_TYPE_ID',
 'MEDIAN_SALE_PRICE',
 'MEDIAN_SALE_PRICE_MOM',
 'MEDIAN_SALE_PRICE_YOY',
 'MEDIAN_LIST_PRICE',
 'MEDIAN_LIST_PRICE_MOM',
 'MEDIAN_LIST_PRICE_YOY',
 'MEDIAN_PPSF',
 'MEDIAN_PPSF_MOM',
 'MEDIAN_PPSF_YOY',
 'MEDIAN_LIST_PPSF',
 'MEDIAN_LIST_PPSF_MOM',
 'MEDIAN_LIST_PPSF_YOY',
 'HOMES_SOLD',
 'HOMES_SOLD_MOM',
 'HOMES_SOLD_YOY',
 'PENDING_SALES',
 'PENDING_SALES_MOM',
 'PENDING_SALES_YOY',
 'NEW_LISTINGS',
 'NEW_LISTINGS_MOM',
 'NEW_LISTINGS_YOY',
 'INVENTORY',
 'INVENTORY_MOM',
 'INVENTORY_YOY',
 'MONTHS_OF_SUPPLY',
 'MONTHS_OF_SUPPLY_MOM',
 'MONTHS_OF_SUPPLY_YOY',
 'MEDIAN_DOM',
 'MEDIAN_DOM_MOM',
 'MEDIAN_DOM_YOY',
 'AVG_SALE_TO_LIST',
 'AVG_SALE_TO_LIST_MOM',
 'AVG_SALE_TO_LIST_YOY',
 'SOLD_ABOVE_LIST',
 'SOLD_ABOVE_LIST_MOM',
 'SOLD_ABOVE_LIST_YOY',
 'PRICE_DROPS',
 'PRICE_DRO

## Clean the data

In [3]:
# drop na data
df = df.dropna().copy()

# convert period start to real date time
df["PERIOD_BEGIN"] = pd.to_datetime(df["PERIOD_BEGIN"])
df = df.sort_values("PERIOD_BEGIN").reset_index(drop=True)

# add new columns for dates
df["MONTH"] = df["PERIOD_BEGIN"].dt.month
df["YEAR"] = df["PERIOD_BEGIN"].dt.year

# seasonal features
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)

# lag features by region
df["PRICE_LAG_1"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(1)
df["PRICE_LAG_3"] = df.groupby("REGION")["MEDIAN_SALE_PRICE"].shift(3)

df = df.drop(columns=["PERIOD_END", "PERIOD_DURATION", "TABLE_ID"])
df = df.dropna().copy()

# encode string columns
categorical_cols = df.select_dtypes(include=["object"]).columns
oe = sk.preprocessing.OrdinalEncoder()
df[categorical_cols] = oe.fit_transform(df[categorical_cols])


## Pre Process before Training

In [4]:
feature_cols = [
    "STATE",
    "CITY",
    "PROPERTY_TYPE",
    "INVENTORY",
    "REGION",
    "HOMES_SOLD",
    "MEDIAN_DOM",
    "AVG_SALE_TO_LIST",
    "PRICE_LAG_1",
    "PRICE_LAG_3",
    "MONTH_SIN",
    "MONTH_COS",
    "YEAR",
]
target_col = "MEDIAN_SALE_PRICE"

# keep the last 3 years for test and the prior 2 years for validation
cutoff_test = df["PERIOD_BEGIN"].max() - pd.DateOffset(years=3)
cutoff_val = cutoff_test - pd.DateOffset(years=2)

train_df = df[df["PERIOD_BEGIN"] < cutoff_val].copy()
val_df = df[
    (df["PERIOD_BEGIN"] >= cutoff_val) & (df["PERIOD_BEGIN"] < cutoff_test)
].copy()
test_df = df[df["PERIOD_BEGIN"] >= cutoff_test].copy()

scaler = sk.preprocessing.MinMaxScaler()

x_train = pd.DataFrame(
    scaler.fit_transform(train_df[feature_cols]),
    columns=feature_cols,
    index=train_df.index,
)
x_val = pd.DataFrame(
    scaler.transform(val_df[feature_cols]),
    columns=feature_cols,
    index=val_df.index,
)
x_test = pd.DataFrame(
    scaler.transform(test_df[feature_cols]),
    columns=feature_cols,
    index=test_df.index,
)

# keep original targets in real dollars for scoring/output
y_train = train_df[target_col].astype("float32")
y_val = val_df[target_col].astype("float32")
y_test = test_df[target_col].astype("float32")

# scale targets for training only
y_scaler = sk.preprocessing.MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).astype(
    "float32"
)
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).astype("float32")
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).astype("float32")

# optional but recommended: convert features to float32 numpy arrays for faster training
x_train_np = x_train.to_numpy(dtype=np.float32)
x_val_np = x_val.to_numpy(dtype=np.float32)
x_test_np = x_test.to_numpy(dtype=np.float32)

y_train_np = y_train.to_numpy(dtype=np.float32)
y_val_np = y_val.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

print("Train shape:", x_train_np.shape, y_train_scaled.shape)
print("Validation shape:", x_val_np.shape, y_val_scaled.shape)
print("Test shape:", x_test_np.shape, y_test_scaled.shape)


Train shape: (1158819, 13) (1158819, 1)
Validation shape: (296235, 13) (296235, 1)
Test shape: (577305, 13) (577305, 1)


## Scoring Method

In [5]:
import math

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score


def scores(y_true, y_pred):
    return {
        "rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "R2_score": r2_score(y_true, y_pred),
        "R": np.corrcoef(np.asarray(y_true).ravel(), np.asarray(y_pred).ravel())[0, 1],
    }


## Build the Model

In [ ]:
# define two-layer neuron configurations and epoch amounts
model_configurations = [
    [10, 5],
    [25, 10],
    [50, 25],
    [75, 50],
    [100, 75],
    [125, 100],
    [200, 125],
    [350, 200],
]

epoch_options = [10, 25, 50, 75, 100, 125, 250, 500]

#setup the model
def build_model(neurons, input_features, learning_rate=0.0001):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(input_features,)),
            tf.keras.layers.Dense(neurons[0], activation="relu"),
            tf.keras.layers.Dense(neurons[1], activation="relu"),
            tf.keras.layers.Dense(1, activation="linear"),
        ]
    )

    model.compile(
        loss="mean_squared_error",
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    )

    return model


## Run Method

In [ ]:
# function to run one model configuration and return results
def run_one_model(neurons, epochs, learning_rate=0.0001, batch_size=4096):
    tf.keras.backend.clear_session()
    model = build_model(
        neurons=neurons,
        input_features=x_train_np.shape[1],
        learning_rate=learning_rate,
    )

    start_time = time.time()
    history = model.fit(
        x_train_np,
        y_train_scaled,
        validation_data=(x_val_np, y_val_scaled),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1,
    )
    elapsed_time = time.time() - start_time

    # predictions come out scaled, so convert back to real dollar values
    y_train_pred_scaled = model.predict(x_train_np, verbose=1)
    y_val_pred_scaled = model.predict(x_val_np, verbose=1)
    y_test_pred_scaled = model.predict(x_test_np, verbose=1)

    y_train_pred = y_scaler.inverse_transform(y_train_pred_scaled).ravel()
    y_val_pred = y_scaler.inverse_transform(y_val_pred_scaled).ravel()
    y_test_pred = y_scaler.inverse_transform(y_test_pred_scaled).ravel()

    # score against original real-dollar targets
    val_scores = scores(y_val_np, y_val_pred)
    test_scores = scores(y_test_np, y_test_pred)

    return {
        "layer_1_neurons": neurons[0],
        "layer_2_neurons": neurons[1],
        "epochs": epochs,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "elapsed_time": elapsed_time,
        "history": history.history,
        "model": model,
        "train_predictions": y_train_pred,
        "val_predictions": y_val_pred,
        "test_predictions": y_test_pred,
        "val_rmse": val_scores["rmse"],
        "val_mape": val_scores["mape"],
        "val_R2_score": val_scores["R2_score"],
        "val_R": val_scores["R"],
        "test_rmse": test_scores["rmse"],
        "test_mape": test_scores["mape"],
        "test_R2_score": test_scores["R2_score"],
        "test_R": test_scores["R"],
    }

# run all the models
def run_all_models(model_configurations, epoch_options, learning_rate=0.001, batch_size=4096):
    all_results = []
    best_result = None
    progress_log_path = output_dir / "progress_log.csv"
    progress_columns = [
        "layer_1_neurons",
        "layer_2_neurons",
        "epochs",
        "learning_rate",
        "batch_size",
        "elapsed_time",
        "val_rmse",
        "val_mape",
        "val_R2_score",
        "val_R",
        "test_rmse",
        "test_mape",
        "test_R2_score",
        "test_R",
    ]
    pd.DataFrame(columns=progress_columns).to_csv(progress_log_path, index=False)

    for neurons in model_configurations:
        for epochs in epoch_options:
            print(
                f"Running model: layer1={neurons[0]}, layer2={neurons[1]}, "
                f"epochs={epochs}, batch_size={batch_size}"
            )

            result = run_one_model(
                neurons=neurons,
                epochs=epochs,
                learning_rate=learning_rate,
                batch_size=batch_size,
            )
            all_results.append(result)

            progress_row = pd.DataFrame(
                [
                    {
                        "layer_1_neurons": result["layer_1_neurons"],
                        "layer_2_neurons": result["layer_2_neurons"],
                        "epochs": result["epochs"],
                        "learning_rate": result["learning_rate"],
                        "batch_size": result["batch_size"],
                        "elapsed_time": result["elapsed_time"],
                        "val_rmse": result["val_rmse"],
                        "val_mape": result["val_mape"],
                        "val_R2_score": result["val_R2_score"],
                        "val_R": result["val_R"],
                        "test_rmse": result["test_rmse"],
                        "test_mape": result["test_mape"],
                        "test_R2_score": result["test_R2_score"],
                        "test_R": result["test_R"],
                    }
                ]
            )
            progress_row.to_csv(progress_log_path, mode="a", header=False, index=False)
            print(f"Saved progress to {progress_log_path}")

            if best_result is None or result["val_rmse"] < best_result["val_rmse"]:
                best_result = result

    results_df = (
        pd.DataFrame(
            [
                {
                    "layer_1_neurons": r["layer_1_neurons"],
                    "layer_2_neurons": r["layer_2_neurons"],
                    "epochs": r["epochs"],
                    "learning_rate": r["learning_rate"],
                    "batch_size": r["batch_size"],
                    "elapsed_time": r["elapsed_time"],
                    "val_rmse": r["val_rmse"],
                    "val_mape": r["val_mape"],
                    "val_R2_score": r["val_R2_score"],
                    "val_R": r["val_R"],
                    "test_rmse": r["test_rmse"],
                    "test_mape": r["test_mape"],
                    "test_R2_score": r["test_R2_score"],
                    "test_R": r["test_R"],
                }
                for r in all_results
            ]
        )
        .sort_values(["layer_1_neurons", "layer_2_neurons", "epochs"])
        .reset_index(drop=True)
    )

    best_by_configuration_df = (
        results_df.sort_values(
            ["layer_1_neurons", "layer_2_neurons", "val_rmse", "epochs"]
        )
        .groupby(["layer_1_neurons", "layer_2_neurons"], as_index=False)
        .first()
        .sort_values(["layer_1_neurons", "layer_2_neurons"])
        .reset_index(drop=True)
    )

    return all_results, results_df, best_by_configuration_df, best_result


## Run the models

In [ ]:
all_results, all_results_df, best_by_configuration_df, best_result = run_all_models(
    model_configurations=model_configurations,
    epoch_options=epoch_options,
    learning_rate=0.0001,
    batch_size=4098,
)
#store all the data
all_results_df.to_csv(output_dir / "all_model_results.csv", index=False)
best_by_configuration_df.to_csv(output_dir / "best_models_by_configuration.csv", index=False)

pd.DataFrame(
    [
        {
            "layer_1_neurons": best_result["layer_1_neurons"],
            "layer_2_neurons": best_result["layer_2_neurons"],
            "epochs": best_result["epochs"],
            "learning_rate": best_result["learning_rate"],
            "batch_size": best_result["batch_size"],
            "elapsed_time": best_result["elapsed_time"],
            "val_rmse": best_result["val_rmse"],
            "val_mape": best_result["val_mape"],
            "val_R2_score": best_result["val_R2_score"],
            "val_R": best_result["val_R"],
            "test_rmse": best_result["test_rmse"],
            "test_mape": best_result["test_mape"],
            "test_R2_score": best_result["test_R2_score"],
            "test_R": best_result["test_R"],
        }
    ]
).to_csv(output_dir / "overall_best_model_summary.csv", index=False)

pd.DataFrame({"y_train": y_train}).to_csv(output_dir / "y_train.csv", index=False)
pd.DataFrame({"y_test": y_test}).to_csv(output_dir / "y_test.csv", index=False)

pd.DataFrame({"train_prediction": best_result["train_predictions"]}).to_csv(
    output_dir / "overall_best_train_predictions.csv", index=False
)
pd.DataFrame({"test_prediction": best_result["test_predictions"]}).to_csv(
    output_dir / "overall_best_test_predictions.csv", index=False
)

for result in all_results:
    run_name = (
        f"layer1_{result['layer_1_neurons']}"
        f"_layer2_{result['layer_2_neurons']}"
        f"_epochs_{result['epochs']}"
    )
    pd.DataFrame(result["history"]).to_csv(
        output_dir / f"{run_name}_history.csv", index=False
    )

for _, row in best_by_configuration_df.iterrows():
    result = next(
        r
        for r in all_results
        if r["layer_1_neurons"] == row["layer_1_neurons"]
        and r["layer_2_neurons"] == row["layer_2_neurons"]
        and r["epochs"] == row["epochs"]
    )
    pd.DataFrame(result["history"]).to_csv(
        output_dir
        / f"best_layer1_{int(row['layer_1_neurons'])}_layer2_{int(row['layer_2_neurons'])}_history.csv",
        index=False,
    )

print(best_by_configuration_df)
print("Overall best model:")
print(
    pd.DataFrame(
        [
            {
                "layer_1_neurons": best_result["layer_1_neurons"],
                "layer_2_neurons": best_result["layer_2_neurons"],
                "epochs": best_result["epochs"],
                "val_rmse": best_result["val_rmse"],
                "test_rmse": best_result["test_rmse"],
            }
        ]
    )
)


Running model: layer1=10, layer2=5, epochs=10, batch_size=4098


I0000 00:00:1774067996.093607  577429 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11028 MB memory:  -> device: 0, name: AMD Radeon RX 6750 XT, pci bus id: 0000:2b:00.0


Epoch 1/10


I0000 00:00:1774067997.436850  577486 service.cc:152] XLA service 0x152ad8003540 initialized for platform ROCM (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774067997.436920  577486 service.cc:160]   StreamExecutor device (0): AMD Radeon RX 6750 XT, AMDGPU ISA version: gfx1030
2026-03-21 00:39:57.455931: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774068000.038665  577486 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


283/283 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - loss: 0.0058 - val_loss: 0.0094
Epoch 2/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9.7189e-04 - val_loss: 0.0026
Epoch 3/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 3.5641e-04 - val_loss: 0.0012
Epoch 4/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 2.1491e-04 - val_loss: 6.9915e-04
Epoch 5/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.5646e-04 - val_loss: 4.7179e-04
Epoch 6/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.2621e-04 - val_loss: 3.5531e-04
Epoch 7/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.0904e-04 - val_loss: 2.8992e-04
Epoch 8/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9.8816e-05 - val_loss: 2.5047e-04
Epoch 9/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 9.2620e-05 - val_loss: 2.2549e-04
Epoch 10/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 8.8840e-05 - val_loss: 2.0924e-04
32373/36214 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step

In [ ]:
# visualization functions
def save_best_rmse_line_plot(best_by_configuration_df, output_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    labels = [
        f"{int(row['layer_1_neurons'])}-{int(row['layer_2_neurons'])}"
        for _, row in best_by_configuration_df.iterrows()
    ]
    ax.plot(
        labels,
        best_by_configuration_df["val_rmse"],
        marker="o",
        linewidth=2.5,
        color="tab:blue",
    )
    ax.set_title("Best Validation RMSE by Two-Layer Configuration")
    ax.set_xlabel("Hidden Layer Configuration")
    ax.set_ylabel("Validation RMSE")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.show()


def save_metric_bar_chart(best_by_configuration_df, metric, ylabel, output_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    labels = [
        f"{int(row['layer_1_neurons'])}-{int(row['layer_2_neurons'])}"
        for _, row in best_by_configuration_df.iterrows()
    ]
    ax.bar(labels, best_by_configuration_df[metric], color="tab:orange")
    ax.set_title(f"Best Two-Layer Models: {ylabel}")
    ax.set_xlabel("Hidden Layer Configuration")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300)
    plt.show()


def save_loss_plots_by_configuration(all_results, output_dir):
    configurations = sorted(
        {(result["layer_1_neurons"], result["layer_2_neurons"]) for result in all_results}
    )

    for layer_1, layer_2 in configurations:
        config_runs = sorted(
            [
                result
                for result in all_results
                if result["layer_1_neurons"] == layer_1
                and result["layer_2_neurons"] == layer_2
            ],
            key=lambda item: item["epochs"],
        )

        fig, ax = plt.subplots(figsize=(10, 6))
        for result in config_runs:
            loss_history = result["history"]["loss"]
            epochs_axis = np.arange(1, len(loss_history) + 1)
            ax.plot(
                epochs_axis,
                loss_history,
                linewidth=2,
                label=f"{result['epochs']} epochs",
            )

        ax.set_title(f"Training Loss for Two-Layer Model ({layer_1}, {layer_2})")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(title="Run length", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(
            output_dir / f"loss_curves_layer1_{layer_1}_layer2_{layer_2}.png",
            dpi=300,
        )
        plt.show()


save_best_rmse_line_plot(
    best_by_configuration_df, output_dir / "best_models_rmse_by_configuration.png"
)
save_metric_bar_chart(
    best_by_configuration_df,
    "val_mape",
    "Validation MAPE",
    output_dir / "best_models_mape_bar.png",
)
save_metric_bar_chart(
    best_by_configuration_df,
    "val_R2_score",
    "Validation R2 Score",
    output_dir / "best_models_r2_bar.png",
)
save_metric_bar_chart(
    best_by_configuration_df,
    "val_R",
    "Validation R",
    output_dir / "best_models_r_bar.png",
)
save_metric_bar_chart(
    best_by_configuration_df,
    "elapsed_time",
    "Elapsed Time (s)",
    output_dir / "best_models_elapsed_time_bar.png",
)
save_loss_plots_by_configuration(all_results, output_dir)
